In [11]:
%pip install openai requests dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from openai import OpenAI
import os
from dotenv import load_dotenv
import json

load_dotenv()

MODEL = "openrouter/free"

prompt = input("Ask Bhondu : ")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("KEY")
)

AGENTS = {
    "general": "Handle general questions from the user",
    "gen": "Generate Image/Video"
}

BOSS_SYSTEM_PROMPT = f"""
You are the problem orchestrator in a multi-agent workflow.

You have the following agents:
{json.dumps(AGENTS, indent=2)}

Your task:
- Read the user request
- Decide which agent should handle it
- Clean up the user request into concise instructions

IMPORTANT:
- Return ONLY raw JSON
- No markdown
- No explanation
- No extra text

Rules:
- General questions -> general
- Image/video generation -> gen
- If unsure -> general

Return ONLY valid JSON in this exact format:

{{
  "Content": "cleaned up request",
  "Agent": "agent_name"
}}
---- END ----
"""

try:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": BOSS_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )

    res = r.choices[0].message.content
    print(res)

    data = json.loads(res)
    
    globals()[data["Agent"]](data["Content"])
    

except Exception as e:
    print("Error:", e)

{
  "Content": "What is the meaning of life?",
  "Agent": "general"
}
prompt


In [ ]:
def general(prompt):
    print("prompt: ", prompt)